In [28]:
from typing import Dict, List, Union

from pydantic import BaseModel
import requests

class URLParams(BaseModel):
    latitude: float
    longitude: float
    start_date: str
    end_date: str
    hourly: Union[str, List[str]]
    timezone: str


api_data_url="https://archive-api.open-meteo.com/v1/archive"

url_params_dict ={
    "latitude": 27,
    "longitude": 30,
    "start_date": "2025-01-30",
    "end_date": "2025-04-30",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "rain",
        "precipitation",
        "cloud_cover",
        "wind_speed_10m",
    ],
    "timezone": "Africa/Cairo",
}

response = requests.get(api_data_url, params=url_params_dict)
if response.status_code == 200:
    data = response.json()
    print("success")

success


In [29]:
import pandas as pd


timestamp = data["hourly"]["time"]
temperature_2m = data["hourly"]["temperature_2m"]
relative_humidity_2m = data["hourly"]["relative_humidity_2m"]
rain = data["hourly"]["rain"]
precipitation = data["hourly"]["precipitation"]
cloud_cover = data["hourly"]["cloud_cover"]
wind_speed_10m = data["hourly"]["wind_speed_10m"]

# combine all features into a single dataframe
# data = pd.DataFrame({
#     "timestamp": timestamp,
#     "temperature_2m": temperature_2m,
#     "relative_humidity_2m": relative_humidity_2m,
#     "rain": rain,
#     "precipitation": precipitation,
#     "cloud_cover": cloud_cover,
#     "wind_speed_10m": wind_speed_10m
# })

data = pd.DataFrame({
    "timestamp": timestamp,
    "temperature_2m": temperature_2m,
})

data.set_index("timestamp", inplace=True)
data.sort_index(inplace=True)
data.index = pd.to_datetime(data.index)

In [30]:
# take average of the temperature_2m for each day
data = data.resample("D").mean()
# take the first 10 days
data.tail(10)

,temperature_2m
timestamp,
2025-04-21,27.875000
2025-04-22,32.929167
2025-04-23,32.891667
2025-04-24,26.358333
2025-04-25,22.612500
2025-04-26,23.012500
2025-04-27,23.204167
2025-04-28,24.258333
2025-04-29,27.137500


In [31]:
from sktime.split import temporal_train_test_split
# Split the data into training and testing sets
train, test = temporal_train_test_split(data, test_size=0.2)

In [ ]:
import pandas as pd
from sktime.forecasting.fbprophet import Prophet
from sktime.forecasting.model_selection import ForecastingGridSearchCV, ExpandingWindowSplitter 
from sktime.forecasting.base import ForecastingHorizon
from sktime.performance_metrics.forecasting import mean_absolute_error

fh = ForecastingHorizon(test.index, is_relative=False)

# Step 3: Define base model and parameter grid
forecaster = Prophet(
)

param_grid = {
    "n_changepoints": [10, 25, 50],
    "seasonality_mode": ["additive", "multiplicative"],
    "changepoint_range": [0.8, 0.9, 0.95],
    "seasonality_prior_scale": [1.0, 5.0, 10.0],
    "changepoint_prior_scale": [0.05, 0.1, 0.5],
    "weekly_seasonality": [True, False],
    "yearly_seasonality": [True, False],
    "daily_seasonality": [True, False],
}

# Step 4: Cross-validation strategy
cv = ExpandingWindowSplitter(fh=1, initial_window=30, step_length=5)

# Step 5: Hyperparameter tuning using grid search
gscv = ForecastingGridSearchCV(
    forecaster=forecaster,
    param_grid=param_grid,
    cv=cv,
    scoring=mean_absolute_error,
)

# Step 6: Fit and predict
gscv.fit(train)
print("✅ Best Parameters:", gscv.best_params_)

y_pred = gscv.predict(fh)
mape = mean_absolute_error(test, y_pred)
print("📉 MAPE on test set:", mape)


KeyboardInterrupt: 

In [ ]:
print("Best Parameters:", gscv.best_params_)
print("Best Score:", gscv.best_score_)
print("Best Forecaster:", gscv.best_forecaster_)